### 1.3.7.6. Jacobian-Vector and Vector-Jacobian Products

$$
\text{JVP:}\quad \mathbf{v} \mapsto J\,\mathbf{v}\ \ (\text{forward}),
\qquad
\text{VJP:}\quad \mathbf{w} \mapsto J^{\top}\mathbf{w}\ \ (\text{reverse}) .
$$

**Explanation:**

Automatic differentiation never forms the full [Jacobian](../05_Multivariable_Differential_Calculus/07_jacobian.ipynb) $J$ — it computes its action on vectors. The Jacobian-vector product $J\mathbf{v}$ pushes a tangent direction $\mathbf{v}$ forward through the map (one [forward-mode](./07_forward_mode_automatic_differentiation.ipynb) sweep), and the vector-Jacobian product $J^\top\mathbf{w}$ pulls a cotangent $\mathbf{w}$ backward (one [reverse-mode](./08_reverse_mode_automatic_differentiation.ipynb) sweep). For a scalar loss the gradient is a single VJP with seed $\mathbf{w} = 1$, which is why reverse mode computes a gradient at the cost of about one function evaluation regardless of input dimension.

**Properties:**
- JVP is cheap when there are few inputs (tall $J$); VJP is cheap when there are few outputs (wide $J$).
- A full $m\times n$ Jacobian costs $n$ JVPs (column by column) or $m$ VJPs (row by row).

**Numerical Example:**

Let $\mathbf{F}(x_1, x_2) = \begin{bmatrix} x_1^2 x_2 \\ \sin x_1 \end{bmatrix}$, with Jacobian $J = \begin{bmatrix} 2x_1x_2 & x_1^2 \\ \cos x_1 & 0 \end{bmatrix}$. At $(1, 2)$, $J = \begin{bmatrix} 4 & 1 \\ \cos 1 & 0 \end{bmatrix}$.

JVP with $\mathbf{v} = [1, 0]^\top$ (the first column of $J$):
$$
J\mathbf{v} = \begin{bmatrix} 4 \\ \cos 1 \end{bmatrix} \approx \begin{bmatrix} 4 \\ 0.540 \end{bmatrix}.
$$

VJP with $\mathbf{w} = [1, 1]^\top$:
$$
J^\top\mathbf{w} = \begin{bmatrix} 4 + \cos 1 \\ 1 \end{bmatrix} \approx \begin{bmatrix} 4.540 \\ 1 \end{bmatrix}.
$$

In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2")
variables = sp.Matrix([x_1, x_2])
vector_map = sp.Matrix([x_1**2 * x_2, sp.sin(x_1)])

jacobian = vector_map.jacobian(variables)
point = {x_1: 1, x_2: 2}
jacobian_at_point = jacobian.subs(point)

tangent = sp.Matrix([1, 0])
cotangent = sp.Matrix([1, 1])
jvp = jacobian_at_point * tangent
vjp = jacobian_at_point.T * cotangent

print("J(1,2) ="); sp.pprint(jacobian_at_point)
print("JVP  J·v =", [sp.N(component, 6) for component in jvp])
print("VJP  Jᵀ·w =", [sp.N(component, 6) for component in vjp])

J(1,2) =
⎡  4     1⎤
⎢         ⎥
⎣cos(1)  0⎦
JVP  J·v = [4.00000, 0.540302]
VJP  Jᵀ·w = [4.54030, 1.00000]


**Equivalent CasADi implementation:**

`ca.jtimes` computes the JVP, and its transposed form (`tr=True`) computes the VJP — the forward and reverse AD primitives, without ever materializing $J$.

In [2]:
import casadi as ca

variables = ca.SX.sym("x", 2)
vector_map = ca.vertcat(variables[0]**2 * variables[1], ca.sin(variables[0]))

jvp = ca.jtimes(vector_map, variables, ca.DM([1, 0]))
vjp = ca.jtimes(vector_map, variables, ca.DM([1, 1]), True)
jvp_function = ca.Function("jvp", [variables], [jvp])
vjp_function = ca.Function("vjp", [variables], [vjp])

print("JVP J·v at (1,2)  =", jvp_function([1, 2]))
print("VJP Jᵀ·w at (1,2) =", vjp_function([1, 2]))

JVP J·v at (1,2)  = [4, 0.540302]
VJP Jᵀ·w at (1,2) = [4.5403, 1]


**References:**

[📘 Griewank, A., & Walther, A. (2008). *Evaluating Derivatives* (2nd ed.). SIAM.](https://epubs.siam.org/doi/book/10.1137/1.9780898717761)  
[📗 Baydin, A. G., Pearlmutter, B. A., Radul, A. A., & Siskind, J. M. (2018). *Automatic Differentiation in Machine Learning: a Survey*. JMLR 18(153).](https://jmlr.org/papers/v18/17-468.html)

---

[⬅️ Previous: Matrix Chain Rule](./05_matrix_chain_rule.ipynb) | [Next: Forward-Mode Automatic Differentiation ➡️](./07_forward_mode_automatic_differentiation.ipynb)